In [22]:
# Cell 1: Imports and file paths
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import joblib  # optional: save model later

# file paths (adjust if necessary)
TRAIN_PATH = "./Dataset/high_salary.csv"
LIVE_PATH = "./Dataset/high_salary.live.csv"

print("Ready. Files expected:", TRAIN_PATH, "and", LIVE_PATH)

Ready. Files expected: ./Dataset/high_salary.csv and ./Dataset/high_salary.live.csv


In [23]:
# Cell 2: Load data and inspect
train = pd.read_csv(TRAIN_PATH)
live = pd.read_csv(LIVE_PATH)

print("Train shape:", train.shape)
print("Live shape:", live.shape)
display(train.head())   # in Jupyter this will render nicely
display(live.head())

Train shape: (20900, 19)
Live shape: (6967, 18)


,id,social-security-number,house-number,age-group,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capitalgain,capitalloss,hoursperweek,native-country-code,native-country,label
0,8616,552701574.0,9854.0,2.0,self-emp-inc,270079.0,bachelors,13.0,married-civ-spouse,exec-managerial,husband,white,male,0.0,0.0,3.0,USA,united-states,1.0
1,21982,956556990.0,7588.0,3.0,local-gov,146325.0,doctorate,16.0,married-civ-spouse,prof-specialty,husband,white,male,0.0,2.0,2.0,USA,united-states,1.0
2,11191,958358623.0,6729.0,0.0,private,240767.0,hs-grad,9.0,never-married,other-service,not-in-family,white,female,0.0,0.0,1.0,USA,united-states,0.0
3,22229,224206693.0,6288.0,2.0,private,118536.0,hs-grad,9.0,divorced,machine-op-inspct,other-relative,black,male,0.0,0.0,2.0,USA,united-states,0.0
4,20732,276413230.0,8276.0,3.0,private,160440.0,bachelors,13.0,married-civ-spouse,sales,husband,white,male,0.0,0.0,3.0,USA,united-states,1.0


,id,social-security-number,house-number,age-group,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capitalgain,capitalloss,hoursperweek,native-country-code,native-country
0,6111,565712576.0,9337.0,0.0,private,287357.0,11th,7.0,married-civ-spouse,protective-serv,husband,white,male,0.0,0.0,2.0,USA,united-states
1,11214,329599477.0,6457.0,1.0,private,167558.0,hs-grad,9.0,never-married,sales,unmarried,white,female,0.0,0.0,1.0,MEX,mexico
2,5554,222432362.0,189.0,4.0,private,27385.0,hs-grad,9.0,married-civ-spouse,exec-managerial,husband,white,male,0.0,0.0,3.0,USA,united-states
3,25131,209882990.0,2048.0,2.0,local-gov,153976.0,masters,14.0,married-civ-spouse,prof-specialty,husband,white,male,0.0,0.0,2.0,USA,united-states
4,14324,936607310.0,8185.0,3.0,private,20956.0,hs-grad,9.0,married-civ-spouse,craft-repair,husband,white,male,0.0,0.0,2.0,USA,united-states


In [24]:
# Cell 3: Basic data inspection
print("Data types:")
print(train.dtypes)

print("\nMissing values in train:")
print(train.isna().sum())

print("\nLabel distribution in training set:")
print(train['label'].value_counts(normalize=True) * 100)

Data types:
id                          int64
social-security-number    float64
house-number              float64
age-group                 float64
workclass                  object
fnlwgt                    float64
education                  object
education-num             float64
marital-status             object
occupation                 object
relationship               object
race                       object
sex                        object
capitalgain               float64
capitalloss               float64
hoursperweek              float64
native-country-code        object
native-country             object
label                     float64
dtype: object

Missing values in train:
id                           0
social-security-number      14
house-number                27
age-group                   13
workclass                 1057
fnlwgt                      23
education                   20
education-num               14
marital-status              23
occupation             

In [25]:
# Cell 4: Split features and target
X = train.drop(columns=['label'])
y = train['label']

# Detect categorical and numerical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['id', 'social-security-number', 'house-number', 'age-group', 'fnlwgt', 'education-num', 'capitalgain', 'capitalloss', 'hoursperweek']
Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country-code', 'native-country']


In [26]:
# Cell 5: Create preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("✅ Preprocessor ready.")

✅ Preprocessor ready.


In [27]:
# Cell 6: Split train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])

Training samples: 16720
Validation samples: 4180


In [28]:
# Cell 7: Build pipeline with RandomForest
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
print("✅ Model trained successfully.")

✅ Model trained successfully.


In [29]:
# Cell 8: Evaluate on validation data
y_val_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1-score:", f1_score(y_val, y_val_pred))
print("\nClassification report:\n", classification_report(y_val, y_val_pred))

Accuracy: 0.816267942583732
Precision: 0.7710511832691249
Recall: 0.7992013690815745
F1-score: 0.7848739495798319

Classification report:
               precision    recall  f1-score   support

         0.0       0.85      0.83      0.84      2427
         1.0       0.77      0.80      0.78      1753

    accuracy                           0.82      4180
   macro avg       0.81      0.81      0.81      4180
weighted avg       0.82      0.82      0.82      4180



In [33]:
# Cell 9: Predict on live dataset and save predictions (fixed version)

# --- Detect ID column (case-insensitive) ---
id_col = None
for col in live.columns:
    if col.lower() == "id":
        id_col = col
        break

# --- Handle ID missing ---
if id_col is None:
    # If no id column, create one based on row index (1, 2, 3, ...)
    live = live.copy()
    live.insert(0, "id", range(1, len(live) + 1))
    id_col = "id"

# --- Drop only non-feature columns (so model gets same features as training) ---
expected_features = [c for c in X.columns]  # from training
live_features = live[expected_features]

# --- Predict ---
live_predictions = model.predict(live_features)

# --- Prepare output dataframe ---
output = pd.DataFrame({
    "id": live[id_col],
    "prediction": live_predictions
})

# --- Save file ---
GROUP_NAME = "G19"  # <-- change this
output_filename = f"{GROUP_NAME}_predictions.live.csv"
output.to_csv(output_filename, index=False)

print(f"✅ File saved as: {output_filename}")
print(f"Rows: {len(output)}")
display(output.head())

✅ File saved as: G19_predictions.live.csv
Rows: 6967


,id,prediction
0,6111,0.0
1,11214,0.0
2,5554,1.0
3,25131,1.0
4,14324,0.0


In [32]:
# Cell 10: Save model (optional)
import joblib
joblib.dump(model, f"{GROUP_NAME}_model.joblib")
print(f"💾 Model saved as {GROUP_NAME}_model.joblib")

💾 Model saved as G19_model.joblib
